**Exemplary load profiles**  
This notebook prepares exemplary demand and generation profiles for the first prototype of Keks.  
The load profiles include:  
- Kritis (Fireforce) from data shared by Stadt Offenburg
- Residential house from the online-tool nPro  
- Industry from the online-tool nPro  

The generation profiles include:  
- per unit values of solar generation based on solar irradiation provided by pvgis  
- Heat pump COP time series, based solely on approximation and author-experience.  

@Author: AqibThenndan

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
os.getcwd()

In [ ]:
import src.utils as utils

In [ ]:
dir = os.path.join(os.getcwd(), 'data')

In [ ]:
#DF to save load profiles of all nodes in proto-network
load_profiles_pu = pd.DataFrame({})

#DF with peak load powers
load_p_max = pd.DataFrame({}, columns = ['P_max(kW)'])

gen_p_max = pd.DataFrame({}, columns = ['P_max(kW)'])

gen_profiles_pu = pd.DataFrame({})

In [ ]:
full_year = pd.date_range(start = '2026-01-01 00:00:00', end = '2026-12-31 23:00:00', freq = 'h')

# Load profiles

## Kritis
Using dataset shared by yamit as source, easier that way

In [ ]:
sog = pd.read_excel(os.path.join(dir,"Stadt_Offenburg_Lastkurven_Stündlich_Detailed.xlsx" ), index_col = 0, sheet_name = 'Feuerwehr', parse_dates = True)
sog.index = sog.index.str.replace(r"\.\d+$", "", regex=True) #one timestamp had microseconds wierdly
sog.index = pd.to_datetime(sog.index)
sog.index = sog.index.map(lambda t: t.replace(year=2026))
sog

In [ ]:
sog = utils.clean_index(snaps = full_year, df = sog)

In [ ]:
sog.isna().sum()

In [ ]:
# capacities of the BHKW and Kessel
gen_p_max.loc['Kritis_chp', 'P_max(kW)'] = np.round(sog['BHKW (kWh).1'].max(), 2)
gen_p_max.loc['Kritis_kessel', 'P_max(kW)'] = np.round(sog['Kessel (kWh)'].max(), 2)

gen_p_max

In [ ]:
load_p_max.loc['kritis_thermal', 'P_max(kW)'] = np.round(sog['Wärmeverbrauch (kWh)'].max(), 2)
load_profiles_pu['kritis_thermal'] = sog['Wärmeverbrauch (kWh)']/load_p_max.loc['kritis_thermal', 'P_max(kW)']

load_p_max.loc['kritis', 'P_max(kW)'] = np.round(sog['Stromverbrauch (kWh)'].max(), 2)
load_profiles_pu['kritis'] = sog['Stromverbrauch (kWh)']/load_p_max.loc['kritis', 'P_max(kW)']

In [ ]:
sog.index[0],sog.index[-1]

In [ ]:
load_profiles_pu

In [ ]:
load_profiles_pu.isna().sum()

## Residential

10 MFHs aggregated  
12 aparments each, 120 apartments in total  
Heated floor area/avg/apartment = 100 sq.m
total heated floor area = 12.000 sq.m  

npro:
SH - 248 kWh/m2/yr  
DHW - 21 kWh/m2/yr  
El - 22 kWh/m2/yr  
 



In [ ]:
import locale

locale.setlocale(locale.LC_TIME, "en_US.UTF-8")

residential = pd.read_excel(os.path.join(dir, "proto_residential.xlsx"), skiprows = 16, usecols = [1,2,3], index_col = 0)
residential_el = pd.read_excel(os.path.join(dir, "proto_residential.xlsx"), sheet_name= 'Electricity', skiprows = 16, usecols = [1,2,3], index_col = 0)

residential.index = pd.to_datetime(residential.index, format = '%a, %d.%m. %H:%M')
residential.index = residential.index.map(lambda t: t.replace(year=2026))

residential_el.index = pd.to_datetime(residential_el.index, format = '%a, %d.%m. %H:%M')
residential_el.index = residential_el.index.map(lambda t: t.replace(year=2026))

residential['Total'] = residential.sum(axis=1)
residential

In [ ]:
len(residential.index)/24

In [ ]:
residential.index

In [ ]:

residential_el = utils.clean_index(snaps = full_year, df = residential_el)
residential = utils.clean_index(snaps = full_year, df = residential)

In [ ]:
residential.isna().sum()

In [ ]:
load_p_max.loc['residential_HT', 'P_max(kW)'] = np.round(residential['Total'].max(), 2)
load_profiles_pu['residential_HT'] = residential['Total']/load_p_max.loc['residential_HT', 'P_max(kW)']

load_p_max.loc['residential', 'P_max(kW)'] = np.round(residential_el['Plug loads (kW)'].max(), 2)
load_profiles_pu['residential'] = residential_el['Plug loads (kW)']/load_p_max.loc['residential', 'P_max(kW)']

In [ ]:
load_profiles_pu['residential_HT'].isna().sum()

In [ ]:
residential.isna().sum(), residential_el.isna().sum()

In [ ]:
residential_el.loc['2026-04-10 10:00:00']/load_p_max.loc['residential', 'P_max(kW)']

In [ ]:
load_profiles_pu.isna().sum()

In [ ]:
load_p_max

### District heating  

Adding two more buses, which would recieve heating supplied a DH network.  
Residential2, & Residential3  
Residential2 aggregates 3 MFHs, while Residential3 aggregates 5MFHs.
The demand profile would be the same as the residential node, the peak load will be adjusted accordinlgly here.  

In [ ]:
type(load_p_max)

In [ ]:
load_p_max.loc['residential', 'P_max(kW)']

In [ ]:
res2 = pd.DataFrame({}, columns = load_p_max.columns)
res2.loc['residential2', 'P_max(kW)'] = load_p_max.loc['residential', 'P_max(kW)']
res2.loc['residential2_LT', 'P_max(kW)'] = residential['Space heating (kW)'].max()
res2.loc['residential2_HT', 'P_max(kW)'] = residential['Domestic hot water (kW)'].max()
res3 = res2.copy(deep = True)
res3.index = res3.index.map(lambda x: x.replace('2', '3'))
res2['P_max(kW)'] *= 3/10 #only 3 MFH buildings
res3['P_max(kW)'] *= 5/10 #only 5 MFH buildings

load_p_max = pd.concat([load_p_max, res2, res3])

# copying over the load profiles
des_load_profiles = pd.DataFrame(index = load_profiles_pu.index)
des_load_profiles['HT'] = residential['Domestic hot water (kW)']/load_p_max.loc['residential2_HT', 'P_max(kW)']
des_load_profiles['LT'] = residential['Space heating (kW)']/load_p_max.loc['residential2_LT', 'P_max(kW)']

load_profiles_pu['residential2_LT'] = des_load_profiles['LT']
load_profiles_pu['residential3_LT'] = des_load_profiles['LT']
load_profiles_pu['residential2_HT'] = des_load_profiles['HT']
load_profiles_pu['residential3_HT'] = des_load_profiles['HT']

load_profiles_pu['residential2'] = load_profiles_pu['residential']
load_profiles_pu['residential3'] = load_profiles_pu['residential']


In [ ]:
des_load_profiles[['LT', 'LT']]

In [ ]:
load_p_max

## Industry
npro:  
5000 m2  
Old construction  
sh: 74 kwh/m2/yr
dhw: 2 kWh/m2/yr  
el: 72 kWh/m2/yr

In [ ]:
locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")

industry = pd.read_excel(os.path.join(dir, "industry_proto.xlsx"), skiprows = 16, usecols = [1,2,3], index_col = 0)

industry.index = pd.to_datetime(industry.index, format = '%a, %d.%m. %H:%M')
industry.index = industry.index.map(lambda t: t.replace(year=2026))
industry['Total'] = industry.sum(axis=1)
# industry
industry_el = pd.read_excel(os.path.join(dir, "industry_proto.xlsx"), sheet_name = "Strom",  skiprows = 16, usecols = [1,2,3], index_col = 0)
industry_el.index = pd.to_datetime(industry_el.index, format = '%a, %d.%m. %H:%M')
industry_el.index = industry_el.index.map(lambda t: t.replace(year=2026))

industry_el

In [ ]:
industry = utils.clean_index(snaps = full_year, df = industry)
industry_el = utils.clean_index(snaps = full_year, df = industry_el)

In [ ]:
load_p_max.loc['industry_thermal', 'P_max(kW)'] = np.round(industry['Total'].max(), 2)
load_profiles_pu['industry_thermal'] = industry['Total']/load_p_max.loc['industry_thermal', 'P_max(kW)']

load_p_max.loc['industry', 'P_max(kW)'] = np.round(industry_el['Nutzerstrom (kW)'].max(), 2)
load_profiles_pu['industry'] = industry_el['Nutzerstrom (kW)']/load_p_max.loc['industry', 'P_max(kW)']

# PV PU generation profile

In [ ]:
import pvlib
import pandas as pd
import numpy as np
import os
import json
from pvlib.pvsystem import PVSystem, Array, FixedMount
from pvlib.modelchain import ModelChain
from pvlib.location import Location

In [ ]:
def fetch_pvgis(lat, long, year, array_configs):
    '''
    Fetches PVGIS hourly poa data for each array, and returns a list of dataframes.
    Additionally, prepares the df with necessary columns and resamples to 15 min with interpolation.
    '''
    weather_list = []
    for arr in array_configs:
        data, input = pvlib.iotools.get_pvgis_hourly(
            latitude=lat,
            longitude=long,
            start=year,
            end=year,
            components=True,
            pvcalculation=False,
            outputformat='csv',
            surface_tilt=arr['tilt'],
            surface_azimuth=180 - arr['azimuth']  # PVGIS: 0=south, -90=east, +90=west
        )

        weather = data[['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse',
                        'temp_air', 'wind_speed']].copy()

        weather['poa_diffuse'] = weather['poa_sky_diffuse'] + weather['poa_ground_diffuse']
        weather['poa_global']  = weather['poa_direct'] + weather['poa_diffuse']

        weather = weather[['poa_global', 'poa_direct', 'poa_diffuse', 'temp_air', 'wind_speed']]
        # weather = weather.resample('15T', offset='10min').interpolate() #there's a 10 min offset in the timestamps of pvgis data
        weather.index = weather.index - pd.Timedelta(minutes=10)
        weather_list.append(weather)

    return weather_list

In [ ]:

#The arrays of pv modules, mutliple possible
arrays_ = [{'tilt' : 30,
            'azimuth' : 180}]
latitude, longitude = 48.45, 7.95


module_info = ['SandiaMod','SunPower_128_Cell_Module__2009__E__']
inverter_info = ['cecinverter', 'AEconversion_GMbH__INV500_90US_xxxxx__208V_']

modules_db = pvlib.pvsystem.retrieve_sam(module_info[0])
module = modules_db[module_info[1]] #replace BAD_CHARS = ' -.()[]:+/",' ; with simply _
# retreiving inverter from db
sapm_inverters = pvlib.pvsystem.retrieve_sam(inverter_info[0])
inverter = sapm_inverters[inverter_info[1]]
#temperature model
temperature_model_parameters = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']

#creates array objects for all array configs, and appends to list arrays
arrays = []
for arr in arrays_:
    arrays.append(Array(
        mount=FixedMount(surface_tilt=arr['tilt'], surface_azimuth=arr['azimuth']), 
        module_parameters=module,
        temperature_model_parameters=temperature_model_parameters,
        modules_per_string=1,
        strings=1
    ))

location = Location(
    latitude,
    longitude,
    name="Stegermatt",
    altitude=110,
    tz="Etc/GMT-1",

)

#getting irradiance data from PVGIS
weather_list = fetch_pvgis(latitude, longitude, 2023, arrays_) #only data till 2023 available
weather = pd.DataFrame(index=weather_list[0].index) #empty df with correct index, required with fetch pvgis

system = PVSystem(arrays=arrays, inverter_parameters=inverter)
mc = ModelChain(system, location)

mc.run_model_from_poa(weather_list)

pv_gen = mc.results.ac
pv_gen.index = pv_gen.index.map(lambda t: t.replace(year=2026)) #replace year to 2026, as the other profiles are for 2026
pv_gen = pv_gen.clip(lower = 0)
# pu profile
## divided by 500 W here, the installed capacity, but max is much lower than this; this needs to be looked into
pv_gen = pv_gen/500 #pu profile

In [ ]:
gen_profiles_pu['pv'] = pv_gen

# HP cop

In [ ]:
365/4

In [ ]:
winter_day = [
    2.5, 2.4, 2.4, 2.3, 2.3, 2.4,
    2.5, 2.6, 2.8, 3.0, 3.2, 3.3,
    3.4, 3.5, 3.5, 3.4, 3.2, 3.0,
    2.9, 2.8, 2.7, 2.6, 2.5, 2.5
]

summer_day = [
    4.2, 4.1, 4.1, 4.0, 4.0, 4.1,
    4.3, 4.5, 4.7, 4.9, 5.1, 5.2,
    5.3, 5.4, 5.4, 5.3, 5.2, 5.0,
    4.8, 4.7, 4.5, 4.4, 4.3, 4.2
]
spring_day = [
    3.3, 3.2, 3.2, 3.1, 3.1, 3.2,
    3.4, 3.6, 3.8, 4.0, 4.2, 4.3,
    4.4, 4.5, 4.5, 4.4, 4.3, 4.1,
    3.9, 3.8, 3.7, 3.5, 3.4, 3.3
]
autumn_day = [
    3.1, 3.0, 3.0, 2.9, 2.9, 3.0,
    3.2, 3.4, 3.6, 3.8, 4.0, 4.1,
    4.2, 4.3, 4.3, 4.2, 4.0, 3.8,
    3.6, 3.5, 3.4, 3.3, 3.2, 3.1
]

cop_profile = (
    winter_day * 62 +
    spring_day * 92 +
    summer_day * 91 +
    autumn_day * 91 +
    winter_day * 29
)
gen_profiles_pu['hp_cop'] = cop_profile


In [ ]:
len(cop_profile)

# Exporting data

## Generation data

In [ ]:
gen_p_max

In [ ]:
gen_profiles_pu

In [ ]:
# gen_profiles = pd.read_excel(os.path.join(dir, "gen_profiles_pu.xlsx"), index_col = 0)
# gen_profiles
# gen_profiles.loc[:, 'Roof_pv'] = gen_profiles_pu['pv']
# gen_profiles.loc[:, 'HP_COP'] = gen_profiles_pu['hp_cop']
gen_profiles_pu = gen_profiles_pu.rename(columns = {
    'time': 'snapshot',
    'pv':'Roof_pv',
    'hp_cop':'HP_COP'
})
gen_profiles_pu.index.name = 'snapshot'
gen_profiles_pu.index = gen_profiles_pu.index.tz_localize(None)
gen_profiles_pu.to_excel(os.path.join(dir, "gen_profiles_pu.xlsx"))

## Loads

In [ ]:
gen_profiles_pu.index[0], gen_profiles_pu.index[-1], len(gen_profiles_pu.index)

In [ ]:
load_profiles_pu.index = load_profiles_pu.index.round('h')
load_profiles_pu.index = load_profiles_pu.index - pd.Timedelta(hours=1) #the load profiles are shifted by 1 hour, so the first value is for 2026-01-01 01:00:00, not 2026-01-01 00:00:00

In [ ]:
load_profiles_pu.index[0], load_profiles_pu.index[-1], len(load_profiles_pu.index)

In [ ]:
pd.date_range('2026-01-01 00:00:00', '2026-12-31 23:00:00', freq = 'h')

In [ ]:
load_profiles_pu.isna().sum()

In [ ]:
load_p_max

In [ ]:
load_profiles_pu.index.name = 'snapshot'
load_profiles_pu = load_profiles_pu.rename(columns = {
    'kritis_electric' : 'kritis',
    'industry_electric' : 'industry',
    'residential_electric' : 'residential',
})

#correcting the order
loads = pd.read_excel(os.path.join(dir, 'loads.xlsx'), index_col = 0)
order = loads.index
load_profiles_pu = load_profiles_pu[order]

#setting the peak load
# load_p_max.rename(index = {'residential_electric': 'residential',
#                            'kritis_electric': 'kritis',
#                            'industry_electric': 'industry'}, inplace = True)
loads['p_max'] = load_p_max['P_max(kW)']/1000 #in MW
loads['p_max'] = np.round(loads['p_max'], 5)

load_profiles_pu.to_excel(os.path.join(dir, "load_profiles_pu.xlsx"))
loads.to_excel(os.path.join(dir, "loads.xlsx"))

In [ ]:
loads

In [ ]:
load_profiles_pu.columns

In [ ]:
order